# DeepSeek OCR Service (Google Colab)

Runs DeepSeek VL2 model on Colab GPU, exposes OCR endpoint via ngrok.

**Usage:**
1. Run all cells in order
2. Copy the printed `DEEPSEEK_OCR_URL` to your `.env` file
3. The local pipeline calls this URL for image-based PDF OCR

**Requirements:** Colab with T4 GPU (free tier works)

In [ ]:
# Cell 1: Install dependencies
!pip install -q flask pyngrok Pillow
!pip install -q transformers==4.45.2 accelerate bitsandbytes timm einops
!pip install -q qwen-vl-utils
print('Dependencies installed')

In [ ]:
# Cell 2: Setup ngrok tunnel
# Get your free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_TOKEN"  # <-- Replace with your token

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
print("ngrok configured")

In [ ]:
# Cell 3: Load Qwen2-VL-2B model (lightweight, works on T4 GPU)
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

print(f"Loading {MODEL_NAME}...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
model.eval()

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f}GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [ ]:
# Cell 4: OCR inference function using Qwen2-VL
from PIL import Image
from qwen_vl_utils import process_vision_info

OCR_PROMPT = (
    "OCR this document image. Extract ALL Vietnamese text exactly as written. "
    "Preserve paragraph structure and line breaks. Output only the extracted text."
)

def run_ocr(image: Image.Image) -> str:
    """Run OCR on a PIL image using Qwen2-VL."""
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": OCR_PROMPT},
            ],
        }
    ]

    # Apply chat template
    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=2048)

    # Decode only new tokens
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    text = processor.decode(generated, skip_special_tokens=True)
    return text.strip()

print("OCR function ready")

In [ ]:
# Cell 4b: Debug — test OCR directly (run after Cell 4)
import base64, io, traceback
from PIL import Image

# Create a simple test image
test_img = Image.new("RGB", (200, 50), "white")

try:
    text = run_ocr(test_img)
    print(f"OCR OK: '{text}'")
except Exception as e:
    traceback.print_exc()
    print(f"\nERROR: {e}")

In [ ]:
# Cell 5: Flask server with /ocr and /health endpoints
import base64
import io
from flask import Flask, request, jsonify

app = Flask(__name__)

@app.route("/ocr", methods=["POST"])
def ocr_endpoint():
    """Accept base64 image, return extracted text.
    
    Request: {"image": "<base64 PNG>"}
    Response: {"text": "<extracted text>"}
    """
    try:
        data = request.json
        if not data or "image" not in data:
            return jsonify({"error": "Missing 'image' field"}), 400

        img_bytes = base64.b64decode(data["image"])
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        text = run_ocr(image)
        return jsonify({"text": text})

    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "model": MODEL_NAME,
        "gpu": torch.cuda.is_available(),
        "gpu_memory_used_gb": round(torch.cuda.memory_allocated() / 1e9, 1),
    })

print("Flask app defined")

In [ ]:
## Troubleshooting

- **DeepSeek VL2 too slow/crashes**: Switched to Qwen2-VL-2B — lighter model (~4GB VRAM), good Vietnamese OCR
- **`LlamaFlashAttention2` error**: Must use `transformers==4.45.2` (Cell 1 handles this)
- **Out of memory**: Restart runtime, ensure T4 GPU is selected
- **ngrok error**: Check your authtoken, free tier allows 1 tunnel
- **Slow inference**: Normal — ~2-5s per page on T4
- **Session disconnects**: Colab free tier disconnects after ~90min idle. Re-run all cells and update URL in `.env`

## Troubleshooting

- **`LlamaFlashAttention2` error**: Must use `transformers==4.45.2` (Cell 1 handles this)
- **`No module deepseek_vl2`**: Restart runtime and re-run Cell 1 first
- **Out of memory**: Restart runtime, ensure T4 GPU is selected. The `tiny` variant needs ~8GB VRAM
- **ngrok error**: Check your authtoken, free tier allows 1 tunnel
- **Slow inference**: Normal — ~2-5s per page on T4
- **Session disconnects**: Colab free tier disconnects after ~90min idle. Re-run all cells and update URL in `.env`